# Transformer

Transformer 是近几年大模型的主要架构基础。它的核心优势主要有两个：一是基于 attention，可以让序列中的 token 直接建立全局依赖；二是相比 RNN 更容易并行训练，可以更充分地利用 GPU 的计算能力。

如果说 RNN 是按照时间步一个 token 一个 token 地向后传递信息，那么 Transformer 更像是让一个序列中的所有 token 同时“互相看见”，再通过 attention score 决定哪些 token 更重要。

## Transformer 架构

Transformer 基于多头注意力机制，下面是其架构示意图。

![Transformer architecture](../../figs/transformer.png)

![Transformer encoder-decoder example](../../figs/transformer2.png)

标准 Transformer 分为 Encoder 和 Decoder 两部分。Encoder 负责编码输入序列，Decoder 负责根据已经生成的上文和 Encoder 的输出继续生成目标序列。

后续很多模型都可以看作 Transformer 的变体。例如 BERT 主要使用 Encoder 结构，适合理解类任务；GPT 主要使用 Decoder-only 结构，适合自回归文本生成；而机器翻译、摘要等 sequence-to-sequence 任务中，也经常使用 Encoder-Decoder 结构。

以文本翻译为例，Encoder 接收原始语言序列，通过 self-attention 得到整段输入的上下文表示：

$$
H_{enc} = Encoder(X_{src})
$$

Decoder 的工作是生成目标语言。它一方面通过 masked self-attention 只能看到当前 token 及其之前的 token；另一方面通过 cross-attention 读取 Encoder 的输出信息。在 cross-attention 中，Decoder hidden states 提供 $Q$，Encoder output 提供 $K$ 和 $V$：

$$
Q = H_{dec}W_Q, \quad K = H_{enc}W_K, \quad V = H_{enc}W_V
$$

这样 Decoder 在生成每个目标 token 时，就既能看到已经生成的目标端上文，也能参考源语言序列的全局信息。

## Transformer 训练

我们首先回顾传统的 RNN 类模型。在一个文本序列样本上进行训练时，模型通常按照 next-token prediction 的方式逐 token 预测。由于 RNN 的 hidden state 依赖前一个时间步，所以即使训练时目标序列已经全部给定，计算过程也很难完全并行，往往需要沿着 sequence length 串行展开。

Transformer 的优势在这里非常明显。因为 self-attention 可以一次性计算整个序列中所有 token 两两之间的注意力分数，所以训练时可以把整个序列同时送入模型。

假设目标序列是：

$$
[y_1, y_2, y_3, y_4, y_5]
$$

训练 Decoder 时，输入会右移一位：

$$
[\mathrm{sos}, y_1, y_2, y_3, y_4]
$$

对应的 ground truth 是：

$$
[y_1, y_2, y_3, y_4, y_5]
$$

也就是说，模型在第 1 个位置基于 $\mathrm{sos}$ 预测 $y_1$，在第 2 个位置基于 $\mathrm{sos}, y_1$ 预测 $y_2$，后续以此类推。

为了防止模型在训练时偷看到未来 token，Decoder 的 self-attention 需要使用 causal mask，也就是一个下三角矩阵：

$$
M_{i,j} =
\begin{cases}
1, & j \le i \\
0, & j > i
\end{cases}
$$

attention score 在 mask 后，当前位置只能对自己和之前位置产生有效注意力，未来位置会被设成一个很小的数，经过 softmax 后权重接近 0。

因此，Transformer 可以在训练时并行计算所有位置的输出，同时又保证每个位置只使用它在真实生成时能够看到的信息。

当然，这种训练方式通常是 teacher forcing：训练时 Decoder 看到的是正确的历史 token，而不是模型自己前一步生成出来的 token。因此它虽然训练效率很高，但也可能带来 exposure bias，也就是训练阶段和推理阶段输入分布不完全一致的问题。

## Transformer 推理过程

Transformer 的训练可以并行，但自回归生成时仍然是串行的。

还是以文本翻译为例，原始语言序列先经过 Encoder 得到：

$$
H_{enc} = Encoder(X_{src})
$$

这个 Encoder 输出在整个生成过程中可以复用。之后维护一个 generated sequence：

1. 第一次生成时，Decoder 输入只有 $\mathrm{sos}$，输出最后一个位置的 logits，通过 vocab projection 和 softmax 得到下一个 token 的概率分布，选出 token1。
2. 第二次生成时，Decoder 输入变成 $[\mathrm{sos}, token1]$，模型继续预测 token2。
3. 后续不断把新 token 拼到序列末尾，直到生成 end token 或达到最大长度。

需要注意，推理时通常不会每次都从头重复计算所有历史 K/V。实际大模型推理中会使用 KV cache，把已经计算过的 Key / Value 缓存起来，从而减少重复计算。不过本 notebook 里的实现是一个简化版，重点是把 Transformer 的整体结构串起来。


下面是 Transformer 的代码实现。这里不包含 tokenizer、词表训练、完整训练循环和 beam search 等工程部分，只实现一个最小的 Encoder-Decoder Transformer 生成模型，帮助理解 Embedding、Positional Encoding、Multi-Head Attention、FeedForward、Add & Norm、causal mask 这些模块是如何组合起来的。


In [14]:
# multi-head attention implementation
import torch
import torch.nn as nn

# basic module for multi-head attention and feedforward
class MultiHeadAttentionWithMask(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.W_qkv = nn.Linear(hidden_dim, hidden_dim * 3)
        self.W_o = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()

        # x: [batch_size, seq_len, hidden_dim]
        qkv = self.W_qkv(x)  # [batch_size, seq_len, hidden_dim * 3]
        q, k, v = qkv.chunk(3, dim=-1)  # each: [batch_size, seq_len, hidden_dim]

        # [batch_size, seq_len, hidden_dim] -> [batch_size, num_heads, seq_len, head_dim]
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # [batch_size, num_heads, seq_len, seq_len]
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            # mask can be [seq_len, seq_len], [batch_size, seq_len, seq_len],
            # or [batch_size, 1, seq_len, seq_len]. It is broadcast across heads.
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            elif mask.dim() != 4:
                raise ValueError("mask must have shape [T,T], [B,T,T], or [B,1,T,T]")
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)  # [batch_size, num_heads, seq_len, seq_len]

        # [batch_size, num_heads, seq_len, head_dim]
        output_heads = torch.matmul(attention_weights, v)

        # [batch_size, num_heads, seq_len, head_dim] -> [batch_size, seq_len, hidden_dim]
        output_concat = output_heads.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
        output = self.W_o(output_concat)

        return output, attention_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.W_q = nn.Linear(hidden_dim, hidden_dim)
        self.W_k = nn.Linear(hidden_dim, hidden_dim)
        self.W_v = nn.Linear(hidden_dim, hidden_dim)
        self.W_o = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, q, k, v, mask=None):
        batch_size, q_len, _ = q.size()
        k_batch_size, kv_len, _ = k.size()
        v_batch_size, v_len, _ = v.size()
        if batch_size != k_batch_size or batch_size != v_batch_size:
            raise ValueError("q, k, and v must have the same batch size")
        if kv_len != v_len:
            raise ValueError("k and v must have the same sequence length")

        # For self-attention, q, k, v come from the same hidden states.
        # For encoder-decoder attention, q comes from decoder, k/v come from encoder.
        q = self.W_q(q)
        k = self.W_k(k)
        v = self.W_v(v)

        # [batch_size, seq_len, hidden_dim] -> [batch_size, num_heads, seq_len, head_dim]
        q = q.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, kv_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, kv_len, self.num_heads, self.head_dim).transpose(1, 2)

        # [batch_size, num_heads, q_len, kv_len]
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            # mask can be [q_len, kv_len], [batch_size, q_len, kv_len],
            # or [batch_size, 1, q_len, kv_len]. It is broadcast across heads.
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            elif mask.dim() != 4:
                raise ValueError("mask must have shape [Q,K], [B,Q,K], or [B,1,Q,K]")
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)  # [batch_size, num_heads, q_len, kv_len]

        # [batch_size, num_heads, q_len, head_dim]
        output_heads = torch.matmul(attention_weights, v)

        # [batch_size, num_heads, q_len, head_dim] -> [batch_size, q_len, hidden_dim]
        output_concat = output_heads.transpose(1, 2).contiguous().view(batch_size, q_len, self.hidden_dim)
        output = self.W_o(output_concat)

        return output, attention_weights

class FeedForward(nn.Module):
    def __init__(self, hidden_dim, ff_dim):
        super().__init__()
        self.linear1 = nn.Linear(hidden_dim, ff_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(ff_dim, hidden_dim)

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))


In [15]:

# Positional Encoding implementation
class PositionalEncoding(nn.Module):
    def __init__(self, hidden_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, hidden_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / hidden_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, hidden_dim]
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [16]:
# Transformer Encoder Layer implementation
class TransformerEncoderLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(hidden_dim, num_heads)
        self.ffn = FeedForward(hidden_dim, ff_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: [batch_size, seq_len, hidden_dim]
        # Multi-head self-attention
        attn_output, _ = self.mha(q=x, k=x, v=x, mask=mask)
        # add & norm
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)

        # Feedforward network
        ffn_output = self.ffn(x)
        # add & norm
        x = x + self.dropout2(ffn_output)
        x = self.norm2(x)

        return x

# Transformer Encoder implementation
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)
    
# Transformer Decoder Layer implementation
class TransformerDecoderLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttentionWithMask(hidden_dim, num_heads)
        self.mha2 = MultiHeadAttention(hidden_dim, num_heads)
        self.ffn = FeedForward(hidden_dim, ff_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
     

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # x: [batch_size, tgt_seq_len, hidden_dim]
        # enc_output: [batch_size, src_seq_len, hidden_dim]
        
        # Masked multi-head attention (decoder self-attention)
        attn_output1, _ = self.mha1(x, tgt_mask)
        # add & norm
        x = x + self.dropout1(attn_output1)
        x = self.norm1(x)

        # Multi-head attention (encoder-decoder cross-attention)
        attn_output2, _ = self.mha2(q=x, k=enc_output, v=enc_output, mask=src_mask)
        # add & norm
        x = x + self.dropout2(attn_output2)
        x = self.norm2(x)

        # Feedforward network
        ffn_output = self.ffn(x)
        # add & norm
        x = x + self.dropout3(ffn_output)
        x = self.norm3(x)

        return x
    
# Transformer Decoder implementation
class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return self.norm(x)

# Transformer model implementation
class Transformer(nn.Module):
    def __init__(self, vocab_size, num_layers, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.src_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.tgt_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.positional_encoding = PositionalEncoding(hidden_dim)
        self.encoder = TransformerEncoder(num_layers, hidden_dim, num_heads, ff_dim, dropout)
        self.decoder = TransformerDecoder(num_layers, hidden_dim, num_heads, ff_dim, dropout)
        self.output_linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src_input, tgt_input, src_mask=None, tgt_mask=None):
        # src_input: [batch_size, src_seq_len]
        # tgt_input: [batch_size, tgt_seq_len]
      
        # Embedding and positional encoding
        src_embedded = self.positional_encoding(self.src_embedding(src_input))
        tgt_embedded = self.positional_encoding(self.tgt_embedding(tgt_input))

        # Encoder
        enc_output = self.encoder(src_embedded, src_mask)

        # Decoder
        dec_output = self.decoder(tgt_embedded, enc_output, src_mask, tgt_mask)

        # Output logits. Softmax is usually applied outside the model when needed.
        logits = self.output_linear(dec_output)  # [batch_size, tgt_seq_len, vocab_size]

        return logits

    def compute_loss(self, src_input, tgt_input, tgt_output, src_mask=None, tgt_mask=None):
        # Forward pass
        logits = self.forward(src_input, tgt_input, src_mask, tgt_mask)
        
        # Compute loss. CrossEntropyLoss expects raw logits, not probabilities.
        loss_fn = nn.CrossEntropyLoss(ignore_index=0)  # Assuming 0 is the padding index
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
        
        return loss

    def generate(self, src_input, max_len, src_mask=None):
        # src_input: [batch_size, src_seq_len]
        batch_size = src_input.size(0)
        generated_seq = torch.zeros(batch_size, max_len, dtype=torch.long).to(src_input.device)
        generated_seq[:, 0] = 1  # Assuming 1 is the start token index

        for t in range(1, max_len):
            tgt_input = generated_seq[:, :t]  # [batch_size, t]
            logits = self.forward(src_input, tgt_input, src_mask)  # [batch_size, t, vocab_size]
            next_token = logits[:, -1, :].argmax(dim=-1)  # [batch_size]
            generated_seq[:, t] = next_token

            if (next_token == 2).all():  # Assuming 2 is the end token index
                break

        return generated_seq


In [17]:
vocab_size = 10000
batch_size = 2
src_len = 10
tgt_len = 10
SOS_ID = 1  # Start of sequence token ID

src_input = torch.randint(2, vocab_size, (batch_size, src_len))
tgt_label = torch.randint(2, vocab_size, (batch_size, tgt_len))
tgt_input = torch.cat([
    torch.full((batch_size, 1), SOS_ID, dtype=tgt_label.dtype),
    tgt_label[:, :-1]
], dim=1)

tgt_mask = torch.tril(torch.ones(tgt_len, tgt_len)).unsqueeze(0).expand(batch_size, -1, -1)  # [batch_size, tgt_len, tgt_len]

model = Transformer(vocab_size=10000, num_layers=6, hidden_dim=512, num_heads=8, ff_dim=2048, dropout=0.1)
y = model(src_input, tgt_input, tgt_mask=tgt_mask)
print(y.shape)  # Expected output shape: [batch_size, seq_len, vocab_size]

torch.Size([2, 10, 10000])
